# Phase 03: ETABS Data Generation Pipeline

This notebook orchestrates the parametric finite element analysis. It is completely **resume-capable** and crash-proof. It generates 5,000 unique damage scenarios, locks them into a master array, and batches the ETABS simulation. If the kernel dies, simply restart and re-run; the pipeline will automatically skip completed batches.

In [2]:
%load_ext autoreload
%autoreload 2
import sys
import os
sys.path.append(os.path.abspath('..'))

from src.etabs_api import start_api

# ---------------------------------------------------------
# SETUP: Connect to ETABS and Define Parameters
# ---------------------------------------------------------
SapModel = start_api(verbose=True)

group_name = "OPT_20"
mat_names = ["C30/37 Zone 1", "C30/37 Zone 2", "C30/37 Zone 3"]
E_i = 32836.6
not_damaged_file = "../data/processed/notDamaged_Flexibility.pkl"
n_modes = 4 # <--- Using optimal elbow point from the convergence analysis!


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Etabs API started successfully


In [ ]:
import numpy as np
import pandas as pd
from src.etabs_api import generate_unique_combinations, run_batches

# --- Configuration ---
n_total_scenarios = 5000
batch_size = 100
master_combo_file = "../data/processed/master_damage_combinations.npy"
output_dir = "../data/processed/batches/"

# ---------------------------------------------------------
# PHASE 1: The Master Damage Array
# ---------------------------------------------------------
if os.path.exists(master_combo_file):
    print(f"Loading existing damage combinations from: {master_combo_file}")
    dataset_array = np.load(master_combo_file)
else:
    print("Generating NEW damage combinations master file...")
    # Generate 5000 unique combos (3 zones, 5% increments)
    dataset_array = generate_unique_combinations(n=n_total_scenarios, step=0.05, n_elements=3)
    np.save(master_combo_file, dataset_array)
    print(f"\u2705 Locked in and saved {n_total_scenarios} combinations to {master_combo_file}")

# ---------------------------------------------------------
# PHASE 2: Start the Resume-Aware Simulation
# ---------------------------------------------------------
print("\nInitiating ETABS Batch Runner...")

generated_files = run_batches(
    dataset_array=dataset_array, 
    batch_size=batch_size, 
    SapModel=SapModel, 
    group_name=group_name, 
    E_i=E_i,
    mat_names=mat_names, 
    not_damaged_file=not_damaged_file, 
    output_dir=output_dir,
    n_modes=n_modes
)

print("\n\U0001F389 ALL BATCHES COMPLETED SUCCESSFULLY!")


Loading existing damage combinations from: ../data/processed/master_damage_combinations.npy

Initiating ETABS Batch Runner...
 Skipping batch 1/50 | File already exists: ../data/processed/batches/batch_001.csv
 Skipping batch 2/50 | File already exists: ../data/processed/batches/batch_002.csv
 Skipping batch 3/50 | File already exists: ../data/processed/batches/batch_003.csv
 Skipping batch 4/50 | File already exists: ../data/processed/batches/batch_004.csv
 Skipping batch 5/50 | File already exists: ../data/processed/batches/batch_005.csv
 Skipping batch 6/50 | File already exists: ../data/processed/batches/batch_006.csv
 Skipping batch 7/50 | File already exists: ../data/processed/batches/batch_007.csv
 Skipping batch 8/50 | File already exists: ../data/processed/batches/batch_008.csv
 Skipping batch 9/50 | File already exists: ../data/processed/batches/batch_009.csv
 Skipping batch 10/50 | File already exists: ../data/processed/batches/batch_010.csv
 Skipping batch 11/50 | File alre

Generating batch: 100%|██████████| 100/100 [2:45:48<00:00, 99.48s/it] 


 Saved ../data/processed/batches/batch_015.csv

 Processing batch 16/50 with 100 samples...


Generating batch: 100%|██████████| 100/100 [3:05:17<00:00, 111.18s/it] 


 Saved ../data/processed/batches/batch_016.csv

 Processing batch 17/50 with 100 samples...


Generating batch: 100%|██████████| 100/100 [3:27:33<00:00, 124.54s/it] 


 Saved ../data/processed/batches/batch_017.csv

 Processing batch 18/50 with 100 samples...


Generating batch: 100%|██████████| 100/100 [3:55:10<00:00, 141.11s/it] 


 Saved ../data/processed/batches/batch_018.csv

 Processing batch 19/50 with 100 samples...


Generating batch:   1%|          | 1/100 [02:35<4:17:03, 155.79s/it]Exception ignored in: <function _compointer_base.__del__ at 0x00000284F3101300>
Traceback (most recent call last):
  File "c:\Users\Z13\Desktop\PFE\pbshm-damage-detection\.venv\Lib\site-packages\comtypes\_post_coinit\unknwn.py", line 276, in __del__
    def __del__(self, _debug=logger.debug) -> None:

KeyboardInterrupt: 
Generating batch:   2%|▏         | 2/100 [05:14<4:16:56, 157.31s/it]

## Phase 3: Final Dataset Assembly
Run this cell *only* when Phase 2 is 100% complete and all batches have been generated.

In [ ]:
# Stitching everything together when the run is done
import glob

all_batch_files = sorted(glob.glob(os.path.join(output_dir, "batch_*.csv")))
df_features = pd.concat((pd.read_csv(f) for f in all_batch_files), ignore_index=True)

# Load the exact labels that align with these rows
labels = np.load(master_combo_file)
df_labels = pd.DataFrame(labels, columns=["Zone1_Sev", "Zone2_Sev", "Zone3_Sev"])

# Combine and save the final AI dataset
final_dataset = pd.concat([df_features, df_labels], axis=1)
final_dataset.to_csv("../data/processed/FINAL_SHM_DATASET.csv", index=False)
print("Final dataset is ready for the Autoencoder!")
